# 🇱🇰 SynhalEES: Colab Slim-Export Helper
⚡ **Free Colab cloud bandwidth — zero home data cost.**

🔄 **Flow:** Cloud Colab ➔ `pull` (heavy 60MB+ files stay on cloud) ➔ `slim-export` (tiny 1KB CSV) ➔ Download slim CSV ➔ Home `slim-import`.

| Step | Execution | Internet Cost |
|---|---|---|
| 📥 Pull (60MB x N tasks) | Google Colab (Free cloud) | **0 MB (Free)** |
| ⚙️ Slim-Export (extract metrics) | Google Colab | **0 MB (Free)** |
| 💾 Download slim CSV | Colab ➔ Your Computer | **~1 KB only** |
| 📊 Slim-Import & Leaderboard Rebuild | Local terminal | **0 MB (Local)** |


## 🔑 1. Kaggle Login & Environment Setup (Run Once)
Clone the repository and install dependencies, then log in via **Colab Secrets** (`KAGGLE_USERNAME` + `KAGGLE_KEY`) or upload your `kaggle.json` token. The last check verifies task access.

In [ ]:
# 🔑 Step 1 | Kaggle login & environment setup
!git clone https://github.com/SynhalaAI/SynhalEES-Benchmark.git
%cd SynhalEES-Benchmark
!pip install -q -e . "kaggle>=2.2.4"  # pin: Colab preinstalls an old kaggle (only push/run); >=2.2.4 has list/status/download
import json as _json, os as _os
from pathlib import Path as _Path
kg = _Path.home() / '.kaggle' / 'kaggle.json'
kg.parent.mkdir(parents=True, exist_ok=True)
saved = False
# Option A (recommended): Colab Secrets KAGGLE_USERNAME + KAGGLE_KEY
try:
    from google.colab import userdata
    _u, _k = userdata.get('KAGGLE_USERNAME'), userdata.get('KAGGLE_KEY')
    kg.write_text(_json.dumps({'username': _u, 'key': _k}))
    kg.chmod(0o600)
    saved = True
    print('kaggle login: Colab Secrets OK (KAGGLE_USERNAME + KAGGLE_KEY)')
except Exception as _e:
    print('Colab Secrets not found (⚠ secrets tab > + Add secret). Falling back to upload...', _e)
# Option B (fallback): upload kaggle.json from Kaggle > Settings > API
if not saved:
    try:
        from google.colab import files
        if not kg.exists():
            print('Upload kaggle.json (Kaggle > Settings > API > Create New Token)')
            up = files.upload()
            for _name, _data in up.items():
                kg.write_bytes(_data)
                break
        kg.chmod(0o600)
        saved = kg.exists()
    except ImportError:
        print('Not on Colab; ensure ~/.kaggle/kaggle.json exists.')
assert saved and kg.exists(), 'kaggle login failed: set Secrets or upload kaggle.json'
print('kaggle.json ready:', str(kg))
!kaggle b t list 2>&1 | head -5


## 📥 2. Pull Artifacts on Cloud (Zero Home Internet Used)
Set `MODE = 'all'` to pull **all 17 benchmark tasks at once**, or `MODE = 'one'` + `TASK` to pull a single pillar. `MODEL` optionally targets one specific model.

In [ ]:
# 📥 Step 2a | Pull config (MODE / TASK / MODEL)
import os
MODE = 'one'  # 'one' = single task below | 'all' = all 17 tasks (pull all)
TASK = 'synhalees-05-sinhala-grammar'  # used when MODE='one'
MODEL = ''  # optional: e.g. 'gemini-3.7-flash' (empty = all models)
os.environ['NB_TASK'] = 'all' if MODE.strip().lower() == 'all' else TASK.strip()
os.environ['NB_MODEL'] = MODEL.strip()
print('pulling:', os.environ['NB_TASK'], os.environ['NB_MODEL'] or '(all models)')


In [ ]:
# 📥 Step 2b | Pull run artifacts from Kaggle
!python -m synhalees kaggle pull "$NB_TASK" $NB_MODEL


## 📦 3. Slim-Export Metrics (Convert 60MB to ~1KB)
Extracts accuracy score, cost, tokens, and latency from `result.json` & `run.json` into lightweight `.slim.csv` files.

In [ ]:
# 📦 Step 3 | Slim-export: metrics 60MB -> ~1KB CSV
!if [ "$NB_TASK" = "all" ]; then for _t in synhalees-01-buddhist-culture synhalees-02-pali-gatha synhalees-03-classical-literature synhalees-04-kavi-sindu synhalees-05-sinhala-grammar synhalees-06-daily-spoken synhalees-07-figurative-sinhala synhalees-08-profanity-nuance synhalees-09-singlish-sms synhalees-10-regional-dialects synhalees-11-astrology-beliefs synhalees-12-general-knowledge synhalees-13-sri-lanka-law synhalees-14-culinary-kitchen synhalees-15-numbers-maths synhalees-vision synhalees-audio; do echo "== $_t =="; python -m synhalees kaggle slim-export "$_t" $NB_MODEL || true; done; else python -m synhalees kaggle slim-export "$NB_TASK" $NB_MODEL; fi
!ls -la kaggle-results/*.slim.csv


## 💾 4. Download Lightweight Slim CSVs (Only KBs to Download)
Downloads only the generated tiny CSVs to your machine.

In [ ]:
# 💾 Step 4 | Download slim CSVs to this machine
from google.colab import files
import glob
for f in sorted(glob.glob('kaggle-results/*.slim.csv')):
    print(f)
    files.download(f)


## 🚀 5. Import & Rebuild Leaderboard on Local PC (No Internet Required)
Run this in your local terminal to import the downloaded slim CSVs directly into the leaderboard without re-downloading anything from Kaggle:

```bash
# Import one or all slim CSVs into submissions/ and automatically rebuild leaderboard data
python -m synhalees kaggle slim-import kaggle-results/synhalees-05-sinhala-grammar.slim.csv
```

💡 **Note (AGENTS.md):** The slim CSV strictly mirrors the `EXT_HEADER` schema (`model,provider,date,pillar,modality,score,cost_usd,tokens,latency_ms`). Re-check notebook cells if the schema ever changes.